# Train Personalized DeepVQE trên Google Colab

Notebook này dùng dataset đã tạo bởi `data_prep_colab.ipynb`, tự chia `metadata.json` thành train/valid manifest, rồi chạy `train_personalized.py`.

## 1. Kiểm tra GPU và mount Google Drive

In [ ]:
import os
import sys
from pathlib import Path

import torch

print(f"Python: {sys.version}")
print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

from google.colab import drive
drive.mount('/content/drive')

## 2. Cài thư viện

Colab thường đã có sẵn `torch` và `torchaudio`; notebook không ép cài `torch==1.11.0` để tránh lỗi không tương thích Python/CUDA mới.

In [ ]:
!pip install -q einops==0.7.0 speechbrain soundfile tqdm

import torchaudio
print(f"Torchaudio: {torchaudio.__version__}")

## 3. Trỏ tới source code và dataset

In [ ]:
# Nếu bạn đặt project trong Google Drive, sửa đường dẫn này cho đúng.
PROJECT_DIR = Path('/content/drive/MyDrive/deepvqe')

# Dataset mặc định khớp với data_prep_colab.ipynb.
DATASET_DIR = Path('/content/drive/MyDrive/TSE_Dataset/mixed_dataset')
METADATA_PATH = DATASET_DIR / 'metadata.json'

# Nếu muốn clone từ GitHub thay vì dùng Drive, điền URL rồi bật USE_GIT_CLONE=True.
USE_GIT_CLONE = False
GIT_REPO_URL = ''

if USE_GIT_CLONE:
    assert GIT_REPO_URL, 'Hãy điền GIT_REPO_URL trước khi clone.'
    clone_dir = Path('/content/deepvqe')
    if not clone_dir.exists():
        !git clone {GIT_REPO_URL} {clone_dir}
    PROJECT_DIR = clone_dir

assert (PROJECT_DIR / 'train_personalized.py').exists(), f'Không thấy train_personalized.py trong {PROJECT_DIR}'
assert METADATA_PATH.exists(), f'Không thấy metadata.json tại {METADATA_PATH}. Hãy chạy data_prep_colab.ipynb trước.'

os.chdir(PROJECT_DIR)
print(f'Project: {PROJECT_DIR}')
print(f'Dataset: {DATASET_DIR}')

## 4. Chia metadata thành train/valid manifest

In [ ]:
import json
import random

SEED = 1337
VALID_RATIO = 0.1
MANIFEST_DIR = DATASET_DIR / 'manifests'
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

with METADATA_PATH.open('r', encoding='utf-8') as f:
    records = json.load(f)

assert isinstance(records, list) and records, 'metadata.json phải là list và không được rỗng.'

rng = random.Random(SEED)
rng.shuffle(records)
valid_size = max(1, int(len(records) * VALID_RATIO))
valid_records = records[:valid_size]
train_records = records[valid_size:]

assert train_records, 'Không đủ dữ liệu train. Hãy tạo thêm sample hoặc giảm VALID_RATIO.'

train_manifest = MANIFEST_DIR / 'train.jsonl'
valid_manifest = MANIFEST_DIR / 'valid.jsonl'

def write_jsonl(path, items):
    with path.open('w', encoding='utf-8') as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

write_jsonl(train_manifest, train_records)
write_jsonl(valid_manifest, valid_records)

print(f'Train samples: {len(train_records)} -> {train_manifest}')
print(f'Valid samples: {len(valid_records)} -> {valid_manifest}')
print('Example record:')
print(json.dumps(train_records[0], ensure_ascii=False, indent=2))

## 5. Cấu hình train

In [ ]:
# phase1_reconstruction: nên chạy trước.
# phase2_speaker_consistency: resume từ phase1/best.pt nếu muốn thêm speaker loss.
# phase3_negative_case: resume từ phase2/best.pt nếu muốn học absent-speaker case.
PHASE = 'phase1_reconstruction'

# Bắt đầu nhỏ để kiểm tra pipeline. Tăng EPOCHS/BATCH_SIZE sau khi chạy ổn.
EPOCHS = 2
BATCH_SIZE = 2
NUM_WORKERS = 2

# True: dùng enrollment wav và ECAPA online, dễ chạy ngay nhưng chậm hơn.
# False: cần manifest có embedding_path .npy.
USE_ONLINE_ECAPA = True

RUN_ROOT = Path('/content/drive/MyDrive/TSE_Dataset/runs')
OUTPUT_DIR = RUN_ROOT / PHASE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# True: mỗi lần chạy lại sẽ tự resume từ OUTPUT_DIR / 'last.pt' nếu file tồn tại.
AUTO_RESUME = True

# Điền đường dẫn checkpoint nếu muốn resume thủ công; giá trị này ưu tiên hơn AUTO_RESUME.
RESUME_FROM = None

print(f'Phase: {PHASE}')
print(f'Output: {OUTPUT_DIR}')

## 6. Chạy train

In [ ]:
import subprocess

device = 'cuda' if torch.cuda.is_available() else 'cpu'

cmd = [
    sys.executable,
    '-u',
    'train_personalized.py',
    '--phase', PHASE,
    '--train-manifest', str(train_manifest),
    '--valid-manifest', str(valid_manifest),
    '--data-root', str(DATASET_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--device', device,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--num-workers', str(NUM_WORKERS),
]

if USE_ONLINE_ECAPA:
    cmd.append('--online-ecapa')

if AUTO_RESUME:
    cmd.append('--auto-resume')

if RESUME_FROM:
    cmd.extend(['--resume', str(RESUME_FROM)])

print(' '.join(cmd))
subprocess.run(cmd, check=True)

## 7. Kiểm tra checkpoint

In [ ]:
for name in ['best.pt', 'last.pt', 'config.json']:
    path = OUTPUT_DIR / name
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f'{name}: {path} ({size_mb:.2f} MB)')
    else:
        print(f'{name}: chưa có')

## 8. Resume phase tiếp theo

In [ ]:
# Ví dụ sau khi phase1 chạy ổn:
# PHASE = 'phase2_speaker_consistency'
# RESUME_FROM = Path('/content/drive/MyDrive/TSE_Dataset/runs/phase1_reconstruction/best.pt')
# OUTPUT_DIR = Path('/content/drive/MyDrive/TSE_Dataset/runs/phase2_speaker_consistency')
# EPOCHS = 40
# BATCH_SIZE = 2
# Sau đó chạy lại cell cấu hình và cell train.